# EDA Starter Notebook

Use this notebook as a starting point for your exploratory data analysis.
Work through each section in order. Cells marked **TODO** are yours to fill in.

---

## 0. Setup & Imports

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from eda_workbench.config import Settings

warnings.filterwarnings("ignore")

# Load project settings from config/settings.yaml
cfg = Settings.from_yaml()

# Apply plot style from config
plt.style.use(cfg.plots.style)
plt.rcParams["figure.figsize"] = cfg.plots.figure_size
plt.rcParams["figure.dpi"] = cfg.plots.dpi

print(f"Random seed : {cfg.experiment.random_seed}")
print(f"Target column: {cfg.data.target_column}")

## 1. Data Loading

Load your dataset from the `data/` directory.
Update `config/settings.yaml` to point at the right file — no code changes needed.

In [ ]:
# TODO: load your dataset
# df = pd.read_csv(cfg.data.train_path)

# For now, create a synthetic example so the notebook runs end-to-end:
rng = np.random.default_rng(cfg.experiment.random_seed)
df = pd.DataFrame(
    {
        "age": rng.integers(18, 65, 200),
        "income": rng.normal(50_000, 15_000, 200).round(2),
        "score": rng.uniform(0, 100, 200).round(1),
        "category": rng.choice(["A", "B", "C"], 200),
        "target": rng.integers(0, 2, 200),
    }
)
# Inject a few missing values to make things interesting
df.loc[rng.choice(200, 10, replace=False), "income"] = np.nan
df.loc[rng.choice(200, 5, replace=False), "score"] = np.nan

print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")

## 2. Quick Look

Get a high-level sense of the data: shape, types, first rows, and summary statistics.

In [ ]:
print(f"Shape: {df.shape}")
df.info()

In [ ]:
df.head(10)

In [ ]:
df.describe(include="all").T

## 3. Missing Values

Identify and visualise where data is absent before doing any analysis.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.concat([missing, missing_pct], axis=1, keys=["count", "pct_%"]).query("count > 0")

In [ ]:
# TODO: decide how to handle missing values
# Options: df.dropna(), df.fillna(df.median(numeric_only=True)), imputation, etc.
pass

## 4. Distributions

Understand the shape of each numerical feature. Look for skew, outliers, and unexpected ranges.

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
print(f"Numerical columns: {num_cols}")

fig, axes = plt.subplots(1, len(num_cols), figsize=(5 * len(num_cols), 4))
for ax, col in zip(axes, num_cols, strict=True):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots to spot outliers
fig, axes = plt.subplots(1, len(num_cols), figsize=(5 * len(num_cols), 4))
for ax, col in zip(axes, num_cols, strict=True):
    sns.boxplot(y=df[col].dropna(), ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 5. Correlations

Measure linear relationships between numerical features. High correlations may indicate redundancy.

In [ ]:
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    ax=ax,
)
ax.set_title("Pearson Correlation Matrix")
plt.tight_layout()
plt.show()

## 6. Feature vs. Target

Examine how each feature relates to the target variable. This is the core of feature understanding.

In [ ]:
target = cfg.data.target_column
feature_cols = [c for c in num_cols if c != target]

fig, axes = plt.subplots(1, len(feature_cols), figsize=(5 * len(feature_cols), 4))
if len(feature_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, feature_cols, strict=True):
    sns.boxplot(x=target, y=col, data=df, ax=ax)
    ax.set_title(f"{col} by {target}")
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs target
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Categorical columns: {cat_cols}")

# TODO: explore categorical distributions against the target variable

## 7. Auto-Profiling

Generate a full HTML report with one line. Open the output file in your browser.

In [ ]:
from pathlib import Path

from ydata_profiling import ProfileReport

output_file = Path(cfg.profiling.output_file)
output_file.parent.mkdir(parents=True, exist_ok=True)

profile = ProfileReport(df, minimal=cfg.profiling.minimal, title="EDA Profile Report")
profile.to_file(output_file)
print(f"Report saved to: {output_file}")

---

## Next Steps

- [ ] Load your real dataset and re-run all cells
- [ ] Handle missing values (Section 3)
- [ ] Investigate outliers (Section 4)
- [ ] Engineer new features based on the correlation analysis (Section 5)
- [ ] Write your findings as functions in `src/eda_workbench/` and add tests in `tests/`